In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")

eigenval_path = 'data/bbj.pca_base.eigenval'
sscore_path = 'data/cteph_agp3k_v6_wgs_merged.sample_qc.variant_qc.bbjproj.sscore'

In [2]:
# STEP0, Data Loading (BBJ + OUR cohort) from scripts/data_loading.py
import importlib
import scripts.data_loading as data_loading_mod

importlib.reload(data_loading_mod)
from scripts.data_loading import (
    DataLoadingConfig,
    load_step0_bbj_and_our,
 )

config_step0 = DataLoadingConfig(
    chunksize=50000,
    bbj_prefix="bbj_",
    verbose=True,
    phenotype_column="PHENO1",
    case_value=2,
    control_value=1,
 )

# Single-entry STEP0 loader: eigenval + BBJ + OUR (+ case/control IID lists)
eigenval, bbj_samples, our_samples, our_case_iids, our_ctrl_iids = load_step0_bbj_and_our(
    eigenval_path=eigenval_path,
    sscore_path=sscore_path,
    config=config_step0,
 )


                        STEP0: DATA LOADING (BBJ + OUR)                         

[CONFIGURATION]
--------------------------------------------------------------------------------
  Eigenvalue path      : data/bbj.pca_base.eigenval
  Score file path      : data/cteph_agp3k_v6_wgs_merged.sample_qc.variant_qc.bbjproj.sscore
  Chunk size           : 50,000
  BBJ prefix           : bbj_
  Phenotype column     : PHENO1
  Case / Control value : 2 / 1

[RESULTS]
--------------------------------------------------------------------------------
  Eigenvalues          : 20 PCs × 4 metrics
  PC1 variance explained: 39.24%
  PC1-2 cumulative var.: 46.70%
  BBJ samples          : 183,013 × 22 cols (49.92 MB)
  OUR samples          : 3,571 × 22 cols (0.94 MB)
  OUR cases / ctrls    : 447 / 3,124



In [3]:
# STEP1, BBJ HDBSCAN Denoising (focused + memory-efficient)
import importlib
import scripts.hdbscan_filtering as hdbscan_filtering_mod
import gc

importlib.reload(hdbscan_filtering_mod)
from scripts.hdbscan_filtering import HDBSCANConfig, run_hdbscan_denoise_bbj

# HDBSCAN is used to remove sparse outliers/noise in PCA space
# and retain stable population structure for downstream modeling.
config_step1 = HDBSCANConfig(
    n_pcs_hdbscan=2,
    use_zscale_hdbscan=True,
    min_cluster_size=50,
    min_samples=6,
    cluster_selection_epsilon=0.005,
    cluster_selection_method="eom",
    metric="euclidean",
    alpha=0.8,
    allow_single_cluster=True,
    leaf_size=40,
    algorithm="best",
    approx_min_span_tree=True,
    gen_min_span_tree=False,
    output_dir="results/01_hdbscan_filtering",
    save_plot=True,
    save_tables=True,
    save_full_table=False,
    verbose=True,
)

bbj_hdbscan = run_hdbscan_denoise_bbj(
    bbj_samples=bbj_samples,
    eigenval=eigenval,
    config=config_step1,
)

# Main downstream input: only non-noise BBJ samples.
bbj_samples_filtered = bbj_hdbscan.bbj_samples_filtered.drop(columns=["HDBSCAN_Label"], errors="ignore")
hdb_summary = bbj_hdbscan.summary

# Release large intermediates after successful run.
del bbj_hdbscan
del bbj_samples
_ = gc.collect()


                         STEP1: HDBSCAN DENOISING (BBJ)                         

[CONFIGURATION]
--------------------------------------------------------------------------------
  n_pcs_hdbscan         : 2
  min_cluster_size      : 50
  min_samples           : 6
  cluster_epsilon       : 0.005
  cluster_method        : eom
  metric                : euclidean
  alpha                 : 0.8
  allow_single_cluster  : True
  leaf_size             : 40
  algorithm             : best
  approx_min_span_tree  : True
  gen_min_span_tree     : False
  use_zscale            : True
  save_plot             : True
  save_full_table       : False
  output_dir            : results/01_hdbscan_filtering

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 183,013
  output_rows           : 181,817
  noise_rows            : 1,196
  noise_ratio           : 0.65%
  clusters_found        : 7


In [4]:
# STEP2, BBJ GMM Clustering (fixed PCs + min BIC)
import importlib
import scripts.gmm_search_audit as gmm_search_audit_mod
import scripts.gmm_clustering as gmm_clustering_mod
import gc

# Reload dependency first, then the main clustering module.
importlib.reload(gmm_search_audit_mod)
importlib.reload(gmm_clustering_mod)
from scripts.gmm_clustering import GMMConfig, run_gmm_fixed_pcs

# Fix the number of PCs to 2, and search for optimal K by BIC.
config_step2 = GMMConfig(
    fixed_n_pcs=2,
    k_min=2,
    k_max=100,
    use_zscale=False,
    covariance_type="full",
    n_init=3,
    init_params="kmeans",
    reg_covar=1e-6,
    max_iter=200,
    random_state=321,
    search_max_samples=200000,
    search_workers=6,
    require_non_empty_clusters=True,
    output_dir="results/02_gmm_clustering",
    save_plot=True,
    save_tables=True,
    verbose=True,
)

gmm_result = run_gmm_fixed_pcs(
    bbj_samples_filtered=bbj_samples_filtered,
    eigenval=eigenval,
    config=config_step2,
)

# Main downstream outputs.
bbj_samples_gmm = gmm_result.bbj_samples_with_cluster
gmm_bic_table = gmm_result.bic_table
gmm_cluster_summary = gmm_result.cluster_summary
gmm_summary = gmm_result.summary

# Keep fitted model for downstream merging/diagnostics.
gmm_model = gmm_result.model

# Release large intermediates after successful run.
del gmm_result
_ = gc.collect()


                   STEP2: GMM CLUSTERING (FIXED PCs, MIN-BIC)                   

[CONFIGURATION]
--------------------------------------------------------------------------------
  fixed_n_pcs           : 2
  k_range               : 2..100
  covariance_type       : full
  n_init                : 3
  init_params           : kmeans
  reg_covar             : 1e-06
  max_iter              : 200
  random_state          : 321
  search_rows           : 181,817
  full_rows             : 181,817
  search_workers        : 6
  require_non_empty     : True
  use_zscale            : False
  output_dir            : results/02_gmm_clustering

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,817
  best_k                : 29
  best_bic              : -2,682,414.87
  non_empty_models      : 99
  clusters_found        : 29


In [5]:
# STEP3, Merge nearby GMM components by Mahalanobis distance + hierarchical clustering
import importlib
import scripts.gmm_component_merging as gmm_component_merging_mod

importlib.reload(gmm_component_merging_mod)
from scripts.gmm_component_merging import GMMComponentMergingConfig, run_gmm_component_merging

config_step3 = GMMComponentMergingConfig(
    merge_threshold=6.0,
    linkage_method="average",
    output_dir="results/03_gmm_component_merging",
    save_plot=True,
    save_tables=True,
    # Panel D: stable and interpretable legend range across runs
    conf_scale_mode="fixed",
    conf_scale_fixed_vmin=0.95,
    conf_scale_fixed_vmax=1.00,
    # Compress differences near 1.0 (high-confidence end)
    conf_norm="power",
    conf_power_gamma=0.40,
    verbose=True,
)

merge_result = run_gmm_component_merging(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_step3,
)

# Common downstream variables (kept for notebook convenience).
merge_map = merge_result.merge_map
label_map = merge_result.label_map
mainland_merged_cluster_id = merge_result.mainland_merged_cluster_id
mainland_premerge_cluster_id = merge_result.mainland_premerge_cluster_id
mainland_premerge_cluster_ids = merge_result.mainland_premerge_cluster_ids


             STEP3: GMM COMPONENT MERGING (MAHALANOBIS + H-CLUSTER)             

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 6.0
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 2
  mainland_premerge_id  : 0
  output_dir            : results/03_gmm_component_merging

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,817
  original_components   : 29
  merged_components     : 6


In [6]:
# STEP3_tmp, alternative merge-threshold run for sensitivity storage
import importlib
import scripts.gmm_component_merging as gmm_component_merging_mod

importlib.reload(gmm_component_merging_mod)
from scripts.gmm_component_merging import GMMComponentMergingConfig, run_gmm_component_merging

config_step3_tmp = GMMComponentMergingConfig(
    merge_threshold=2.5,
    linkage_method="average",
    output_dir="results/03_gmm_component_merging/STEP3_tmp",
    save_plot=True,
    save_tables=True,
    # Keep visualization settings aligned with STEP3 for comparability.
    conf_scale_mode="fixed",
    conf_scale_fixed_vmin=0.95,
    conf_scale_fixed_vmax=1.00,
    conf_norm="power",
    conf_power_gamma=0.40,
    verbose=True,
)

merge_result_step3_tmp = run_gmm_component_merging(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_step3_tmp,
)

# Expose STEP3_tmp outputs without overriding main STEP3 variables.
merge_map_step3_tmp = merge_result_step3_tmp.merge_map
label_map_step3_tmp = merge_result_step3_tmp.label_map
mainland_merged_cluster_id_step3_tmp = merge_result_step3_tmp.mainland_merged_cluster_id
mainland_premerge_cluster_id_step3_tmp = merge_result_step3_tmp.mainland_premerge_cluster_id
mainland_premerge_cluster_ids_step3_tmp = merge_result_step3_tmp.mainland_premerge_cluster_ids


             STEP3: GMM COMPONENT MERGING (MAHALANOBIS + H-CLUSTER)             

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 2.5
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 4
  mainland_premerge_id  : 3
  output_dir            : results/03_gmm_component_merging/STEP3_tmp

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,817
  original_components   : 29
  merged_components     : 13


In [7]:
# STEP4, OUR cohort assignment to pre-merge GMM components
import importlib
from typing import Any, cast
import scripts.our_assignment as our_assignment_mod

importlib.reload(our_assignment_mod)
from scripts.our_assignment import OURAssignmentConfig, run_our_assignment_to_merged_gmm

# Identity map: each original GMM component maps to itself (pre-merge assignment).
n_components_premerge = int(getattr(cast(Any, gmm_model), "n_components"))
premerge_label_map = {int(k): int(k) for k in range(n_components_premerge)}

config_step4 = OURAssignmentConfig(
    output_dir="results/04_our_assignment",
    save_plot=True,
    save_tables=True,
    output_file="our_posterior_probabilities_premerge.tsv",
    figure_file="our_assignment_premerge.png",
    case_label="CTEPH",
    control_label="AGP3K",
    bbj_alpha=0.20,
    verbose=True,
 )

step4_out = run_our_assignment_to_merged_gmm(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    label_map=premerge_label_map,
    merge_map=merge_map,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    training_use_zscale=config_step2.use_zscale,
    config=config_step4,
)

# Mainland-only all-PC KDE visualization and sample export.
import scripts.mainland_all_pcs_kde as mainland_all_pcs_kde_mod

importlib.reload(mainland_all_pcs_kde_mod)
from scripts.mainland_all_pcs_kde import MainlandAllPCsKDEConfig, run_mainland_all_pcs_kde

config_step4_mainland = MainlandAllPCsKDEConfig(
    output_dir="results/04_our_assignment",
    save_plot=True,
    save_tables=True,
    output_file="mainland_samples.fid_iid.txt",
    figure_file="mainland_all_pcs_kde.png",
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    bbj_color="#1F78B4",
    case_color="#E31A1C",
    alpha=0.65,
    verbose=True,
)

mainland_kde_out = run_mainland_all_pcs_kde(
    df_results=step4_out.df_results,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    mainland_cluster_ids=mainland_premerge_cluster_ids,
    eigenval=eigenval,
    step4_config=config_step4,
    config=config_step4_mainland,
)

# Common downstream variables (kept for notebook convenience).
df_results = step4_out.df_results
probs_premerge_our = step4_out.probs_merged_our
assigned_premerge = step4_out.assigned_merged
assignment_conf = step4_out.assignment_confidence
cluster_stats = step4_out.cluster_stats
mainland_samples_fid_iid = mainland_kde_out.fid_iid_file
mainland_all_pcs_figure = mainland_kde_out.figure_png
mainland_all_pcs_df = mainland_kde_out.df_mainland

# Backward-compatible aliases for downstream cells/notebook history.
probs_merged_our = probs_premerge_our
assigned_merged = assigned_premerge


              STEP4: COHORT ASSIGNMENT TO PRE-MERGE GMM COMPONENTS              

[CONFIGURATION]
--------------------------------------------------------------------------------
  output_dir            : results/04_our_assignment
  save_tables           : True
  save_plot             : True
  show_plot             : False

[RESULTS]
--------------------------------------------------------------------------------
  cohort rows           : 3,571
  assigned_clusters (K) : 29
  assignment_tsv        : results/04_our_assignment/our_posterior_probabilities_premerge.tsv
  mainland_samples_tsv  : results/04_our_assignment/mainland_samples.fid_iid.txt
  mainland_cluster_rank : results/04_our_assignment/mainland_samples_cluster_rank.tsv
>>> ANALYZING ALL PC DISTRIBUTIONS FOR MAINLAND SAMPLES...
   -> output_dir = results/04_our_assignment
   -> mainland clusters = [0, 2, 3, 5, 6, 8, 9, 10, 12, 14, 15, 17, 18, 19, 20, 21, 24, 25, 26, 27]
   -> Mainland samples: 3104 / 3571
   -> Case samples (M

In [8]:
# STEP4_tmp, mainland rank-cumulative analysis (Rank1 -> Rank17)
# Per-cluster ranking uses direct pre-merge MAP counts (Assigned_Merged_Cluster) for sorting.
# Cumulative trade-off metrics use composite posterior recomputation (same as STEP5):
# at each k, top-k clusters are merged into one group, posteriors re-normalized, assignment via argmax.
import importlib
import scripts.step4_tmp_mainland_rank_progression as step4_tmp_mod

importlib.reload(step4_tmp_mod)
from scripts.step4_tmp_mainland_rank_progression import (
    Step4TmpMainlandRankProgressionConfig,
    run_step4_tmp_mainland_rank_progression,
 )

config_step4_tmp = Step4TmpMainlandRankProgressionConfig(
    output_dir="results/04_our_assignment/STEP4_tmp",
    max_rank=17,
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    forced_recommended_rank=9,  # override Pareto-auto detection; set None to use auto
    save_plot=True,
    show_plot=False,
    verbose=True,
 )

step4_tmp_out = run_step4_tmp_mainland_rank_progression(
    df_results=step4_out.df_results,
    merge_map=merge_map,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    gmm_model=gmm_model,
    gmm_summary=gmm_summary,
    config=config_step4_tmp,
 )

# Expose STEP4_tmp outputs
step4_tmp_rank_table = step4_tmp_out.rank_table
step4_tmp_cumulative_metrics = step4_tmp_out.cumulative_metrics
step4_tmp_decision_table = step4_tmp_out.decision_table
step4_tmp_recommended_rank = step4_tmp_out.recommended_rank
step4_tmp_output_dir = step4_tmp_out.output_dir
step4_tmp_rank_table_path = step4_tmp_out.rank_table_path
step4_tmp_cumulative_table_path = step4_tmp_out.cumulative_table_path
step4_tmp_decision_table_path = step4_tmp_out.decision_table_path
step4_tmp_figure_path = step4_tmp_out.figure_path



                        STEP4_tmp: MAINLAND RANK-CUMULATIVE ANALYSIS                        
Mainland clusters ranked (top 17): [27, 14, 3, 20, 0, 21, 17, 19, 5, 18, 26, 10, 25, 8, 24, 2, 9]
Recommended rank k   : 9  (forced)
Rank table saved      : results/04_our_assignment/STEP4_tmp/mainland_cluster_rank_table.tsv
Cumulative table saved: results/04_our_assignment/STEP4_tmp/mainland_rank_cumulative_metrics.tsv
Decision table saved  : results/04_our_assignment/STEP4_tmp/mainland_rank_decision_table.tsv
Figure saved          : results/04_our_assignment/STEP4_tmp/mainland_rank_progression_metrics.png
--------------------------------------------------------------------------------------------
 Included_Max_Rank  Included_Cluster_Count                            Included_Clusters  CTEPH_Count  AGP3K_Count  Case_Control_Ratio  Total_Count   GWAS_Neff  PC12_AllSample_Heterogeneity  Delta_Neff  Delta_Heterogeneity  Neff_Gain_per_Heterogeneity  Neff_Norm  Heterogeneity_Norm  Utility_NeffMinus

In [9]:
# STEP5, Mainland subcluster global posterior reassignment
import importlib
import scripts.customize_cluster_assignment as mainland_subcluster_mod

importlib.reload(mainland_subcluster_mod)
from scripts.customize_cluster_assignment import (
    MainlandSubclusterReassignmentConfig,
    run_mainland_subcluster_reassignment,
 )

# Select the top-k mainland clusters by case/ctrl ratio (from STEP4_tmp rank table),
# then merge them into a single composite group and recompute global posteriors.
# Assignment is based on argmax of the recomputed merged posterior.
_step5_included_rank = 17  # include top-n ranked mainland clusters
_step5_top_k_clusters = set(
    int(v) for v in step4_tmp_rank_table.loc[
        step4_tmp_rank_table["Rank"] <= _step5_included_rank, "Cluster"
    ]
)
_step5_all_mainland = set(int(v) for v in mainland_premerge_cluster_ids)
_step5_exclude_ids = tuple(sorted(_step5_all_mainland - _step5_top_k_clusters))

config_step5 = MainlandSubclusterReassignmentConfig(
    output_dir="results/05_customize_cluster_assignment",
    save_plot=True,
    save_tables=True,
    output_file="our_posterior_probabilities_customize_merged.tsv",
    figure_file="our_assignment_customize_merged.png",
    custom_group_label="Mainland Subcluster",
    exclude_cluster_ids=_step5_exclude_ids,
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    bbj_alpha=config_step4.bbj_alpha,
    verbose=True,
 )

step5_out = run_mainland_subcluster_reassignment(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    mainland_premerge_cluster_ids=mainland_premerge_cluster_ids,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_step5,
)

# Common downstream variables.
mainland_subcluster_component_ids = step5_out.customize_cluster
df_results_step5 = step5_out.df_results
posterior_step5 = step5_out.probs_customize
assigned_component_if_not_subcluster = step5_out.assigned_customize
assignment_confidence_step5 = step5_out.assignment_confidence
group_stats_step5 = step5_out.cluster_stats

# Backward-compatible aliases.
customize_cluster = mainland_subcluster_component_ids
df_results_customize = df_results_step5
probs_customize = posterior_step5
assigned_customize = assigned_component_if_not_subcluster
assignment_conf_customize = assignment_confidence_step5
cluster_stats_customize = group_stats_step5



                    STEP5: MAINLAND SUBCLUSTER REASSIGNMENT                     
  mainland_subcluster_ids: [0, 2, 3, 5, 8, 9, 10, 14, 17, 18, 19, 20, 21, 24, 25, 26, 27]
  remaining_component_ids: [1, 4, 6, 7, 11, 12, 13, 15, 16, 22, 23, 28]
  excluded_cluster_ids   : [6, 12, 15]
  output_dir             : results/05_customize_cluster_assignment
  assignment_tsv         : results/05_customize_cluster_assignment/our_posterior_probabilities_customize_merged.tsv


In [10]:
# STEP6, mainland_subcluster-only visualization + unfiltered FID/IID export
import importlib
import scripts.mainland_subcluster_only as step6_mod

importlib.reload(step6_mod)
from scripts.mainland_subcluster_only import (
    MainlandSubclusterOnlyConfig,
    run_mainland_subcluster_only,
 )

config_step6 = MainlandSubclusterOnlyConfig(
    output_dir="results/06_mainland_subcluster_only",
    sample_id_file="mainland_subcluster_samples.fid_iid.txt",
    figure_file="mainland_subcluster_only.png",
    mainland_group_label=config_step5.custom_group_label,
    assigned_group_col="Assigned_Mainland_Subcluster_Group",
    confidence_col="Assignment_Confidence",
    case_label=config_step5.case_label,
    control_label=config_step5.control_label,
    mainland_group_color=config_step5.custom_group_color,
    bbj_color=config_step5.bbj_color,
    bbj_alpha=config_step5.bbj_alpha,
    our_point_size=config_step5.our_point_size,
    save_plot=True,
    show_plot=False,
    verbose=True,
 )

step6_out = run_mainland_subcluster_only(
    df_results_step5=df_results_step5,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    bbj_samples_gmm=bbj_samples_gmm,
    eigenval=eigenval,
    config=config_step6,
)

# Mainland-subcluster all-PC KDE visualization and sample export.
import scripts.mainland_subcluster_all_pcs_kde as mainland_subcluster_all_pcs_kde_mod

importlib.reload(mainland_subcluster_all_pcs_kde_mod)
from scripts.mainland_subcluster_all_pcs_kde import MainlandSubclusterAllPCsKDEConfig, run_mainland_subcluster_all_pcs_kde

config_step6_mainland = MainlandSubclusterAllPCsKDEConfig(
    output_dir="results/06_mainland_subcluster_only",
    save_plot=True,
    save_tables=True,
    output_file="mainland_subcluster_samples.fid_iid.txt",
    figure_file="mainland_subcluster_all_pcs_kde.png",
    mainland_group_label=config_step5.custom_group_label,
    assigned_group_col="Assigned_Mainland_Subcluster_Group",
    confidence_col="Assignment_Confidence",
    case_label=config_step5.case_label,
    control_label=config_step5.control_label,
    bbj_color="#1F78B4",
    case_color="#E31A1C",
    alpha=0.65,
    verbose=True,
)

mainland_subcluster_kde_out = run_mainland_subcluster_all_pcs_kde(
    df_results_step5=df_results_step5,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step6_mainland,
)

# Expose Step6 outputs
df_mainland_subcluster = step6_out.df_mainland_subcluster
df_results_step6_mainland_only = step6_out.df_mainland_subcluster
mainland_subcluster_fid_iid = mainland_subcluster_kde_out.fid_iid_file
mainland_subcluster_all_pcs_figure = mainland_subcluster_kde_out.figure_png
mainland_subcluster_all_pcs_df = mainland_subcluster_kde_out.df_subcluster


                        STEP6: MAINLAND SUBCLUSTER ONLY                         
  assigned_group_col : Assigned_Mainland_Subcluster_Group
  mainland_label     : Mainland Subcluster
  rows (unfiltered)  : 2902
  unlabeled rows     : 0
  sample_id_file     : results/06_mainland_subcluster_only/mainland_subcluster_samples.fid_iid.txt
  figure_file        : results/06_mainland_subcluster_only/mainland_subcluster_only.png
>>> ANALYZING ALL PC DISTRIBUTIONS FOR MAINLAND_SUBCLUSTER SAMPLES...
   -> output_dir = results/06_mainland_subcluster_only
   -> mainland label = Mainland Subcluster
   -> mainland_subcluster samples = 2902 / 3571
   -> Case samples: 434
   -> Control samples: 2468
   -> Total PCs to analyze: 20
   -> Running statistical tests...
   -> Applying FDR correction (Benjamini-Hochberg method) to all tests...
   -> FDR correction complete.
      • Significant by t-test: 5 PC(s)
      • Significant by Mann-Whitney U: 6 PC(s)


In [11]:
# STEP7, mainland_subcluster confidence distribution analysis
import importlib
import scripts.mainland_subcluster_confidence_distribution as step7_mod

importlib.reload(step7_mod)
from scripts.mainland_subcluster_confidence_distribution import (
    run_mainland_subcluster_confidence_distribution,
    MainlandSubclusterConfidenceDistributionConfig,
)

config_step7 = MainlandSubclusterConfidenceDistributionConfig(
    output_dir="results/07_mainland_subcluster_confidence_distribution",
    case_label=str(config_step4.case_label),
    control_label=str(config_step4.control_label),
    save_plot=True,
    show_plot=False,
    verbose=True,
)

step7_out = run_mainland_subcluster_confidence_distribution(
    df_mainland_subcluster=df_mainland_subcluster,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step7,
)

# Expose key outputs
summary_stats_step7 = step7_out.summary_stats
summary_by_group_step7 = step7_out.summary_by_group
step7_figure_path = step7_out.figure_path
step7_outdir = step7_out.output_dir


               STEP7: MAINLAND SUBCLUSTER CONFIDENCE DISTRIBUTION               

  Total samples (Case+Control): 2902
  CTEPH       :   434
  AGP3K       :  2468

  Output directory: results/07_mainland_subcluster_confidence_distribution
  Figure saved   : results/07_mainland_subcluster_confidence_distribution/mainland_subcluster_confidence_distribution.png

--------------------------------------------------------------------------------

Metric                      CTEPH                AGP3K
--------------------------------------------------------------------------------
mean                     0.995180             0.974498
std                      0.028109             0.084618
min                      0.520953             0.390273
25%                      0.999462             0.998836
50%                      0.999989             0.999985
75%                      1.000000             0.999999
max                      1.000000             1.000000



In [12]:
# STEP8, confidence-threshold sensitivity for mainland_subcluster
import importlib
import scripts.mainland_subcluster_confidence_threshold_screening as step8_mod

importlib.reload(step8_mod)
from scripts.mainland_subcluster_confidence_threshold_screening import (
    MainlandSubclusterConfidenceThresholdScreeningConfig,
    run_mainland_subcluster_confidence_threshold_screening,
 )

case_min_conf_step8 = float(
    pd.to_numeric(
        df_mainland_subcluster.loc[
            df_mainland_subcluster["IID"].astype(str).isin(set(str(x) for x in our_case_iids)),
            "Assignment_Confidence",
        ],
        errors="coerce",
    ).min()
 )

config_step8 = MainlandSubclusterConfidenceThresholdScreeningConfig(
    output_dir="results/08_mainland_subcluster_confidence_screening",
    fixed_thresholds=(0.80, 0.85, 0.90, 0.95, 0.99),
    include_case_min_threshold=True,
    case_label=str(config_step4.case_label),
    control_label=str(config_step4.control_label),
    save_plot=True,
    show_plot=False,
    verbose=True,
 )

step8_out = run_mainland_subcluster_confidence_threshold_screening(
    df_mainland_subcluster=df_mainland_subcluster,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step8,
 )

# Expose key outputs
step8_thresholds = step8_out.thresholds
step8_summary_table = step8_out.summary_table
step8_figure_path = step8_out.figure_path
step8_table_path = step8_out.table_path
step8_outdir = step8_out.output_dir


                         STEP8: CONFIDENCE THRESHOLD SCREENING                          

  Base sample sizes: CTEPH=434, AGP3K=2468
  Thresholds used  : 0.520953, 0.800000, 0.850000, 0.900000, 0.950000, 0.990000
  Table saved      : results/08_mainland_subcluster_confidence_screening/mainland_subcluster_confidence_threshold_screening.tsv
  Figure saved     : results/08_mainland_subcluster_confidence_screening/mainland_subcluster_confidence_threshold_screening.png

----------------------------------------------------------------------------------------
 threshold  case_total  control_total  case_kept  control_kept  case_removed  control_removed  case_keep_rate  control_keep_rate  case_removed_rate  control_removed_rate
  0.520953         434           2468        434          2446             0               22        1.000000           0.991086           0.000000              0.008914
  0.800000         434           2468        432          2345             2              123       

In [13]:
# STEP9, threshold-based FID/IID export (script + config + runner)
import importlib
import scripts.mainland_subcluster_threshold_sample_export as step9_mod

importlib.reload(step9_mod)
from scripts.mainland_subcluster_threshold_sample_export import (
    MainlandSubclusterThresholdSampleExportConfig,
    run_mainland_subcluster_threshold_sample_export,
 )

config_step9 = MainlandSubclusterThresholdSampleExportConfig(
    output_dir="results/09_threshold_sample_exports",
    summary_file="threshold_retained_removed_summary.tsv",
    fixed_thresholds=(0.80, 0.85, 0.90, 0.95, 0.99),
    include_case_min_threshold=True,
    case_label=str(config_step4.case_label),
    control_label=str(config_step4.control_label),
    fid_col="FID",
    iid_col="IID",
    confidence_col="Assignment_Confidence",
    export_case_ctrl_files=False,
    verbose=True,
 )

step9_out = run_mainland_subcluster_threshold_sample_export(
    df_mainland_subcluster=df_mainland_subcluster,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step9,
 )

# Expose Step9 outputs
step9_default_threshold = step9_out.default_threshold
step9_thresholds = step9_out.thresholds
step9_summary = step9_out.summary_table
step9_outdir = step9_out.output_dir
step9_summary_path = step9_out.summary_table_path


                           STEP9: THRESHOLD-BASED FID/IID EXPORT                            
Default threshold (case min): 0.520953
Thresholds: 0.5210, 0.8000, 0.8500, 0.9000, 0.9500, 0.9900
Output directory: results/09_threshold_sample_exports
Summary table   : results/09_threshold_sample_exports/threshold_retained_removed_summary.tsv
------------------------------------------------------------
 threshold  case_total  ctrl_total  case_retained  ctrl_retained
  0.520953         434        2468            434           2446
  0.800000         434        2468            432           2345
  0.850000         434        2468            432           2317
  0.900000         434        2468            429           2289
  0.950000         434        2468            425           2213
  0.990000         434        2468            402           2052
